# 2.11.3 Gradient Boosting

## Setup

In [1]:
import os 
import warnings
import librosa
import numpy as np
import pandas as pd
from scipy.io import wavfile

import cufflinks as cf
import IPython.display as ipd
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import plotly.figure_factory as ff
from sklearn.ensemble import GradientBoostingClassifier
from sklearn import set_config
from sklearn.metrics import (accuracy_score, confusion_matrix,make_scorer,recall_score)
from sklearn.model_selection import (train_test_split, RepeatedKFold, RepeatedStratifiedKFold, 
                                     cross_validate, cross_val_score, KFold)
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval

pd.set_option("display.max_columns", 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
pio.templates.default = "plotly_dark"
set_config(transform_output="pandas")
set_config(display='diagram')
warnings.filterwarnings("ignore")

%matplotlib inline

# Directorio de datos
dir_data = '../03_Data/'

In [2]:
def get_cv_scores_report_classification(estimator, X, y, n_splits):
    cv_scores = cross_validate(
                    estimator = estimator,
                    X         = X,
                    y         = y,
                    scoring   = 'accuracy',
                    cv        = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=5, random_state=333),
                )
    
    accuracy_scores = cv_scores['test_score']
    mean_accuracy = accuracy_scores.mean()
    std_accuracy = accuracy_scores.std()
    
    # Impresión de los resultados
    print(f"Accuracy: {mean_accuracy:.4f} (+/- {2*std_accuracy:.4f})")
def classification_metrics(X, y, estimator):
    ls_scores_roc = cross_val_score(estimator=estimator, X=X, y=y, scoring="accuracy", n_jobs=-1, cv=4)
    print(f"Accuracy media: {np.mean(ls_scores_roc):,.2f}, desviación estándar: {np.std(ls_scores_roc)}")
# Función que reproduce sonido en Jupyter
def wavPlayer(filepath):
    rate, data = wavfile.read(filepath)
    display(pd.Series(data).iplot())
    return ipd.Audio(filepath, autoplay=True)
# Función para visualizar el mapa de calor de las frecuencias de sonido
def plot_heatmap(data):
    X = librosa.stft(data)
    Xdb = librosa.amplitude_to_db(abs(X))
    plt.figure(figsize=(14, 5))
    librosa.display.specshow(Xdb, sr=sr, x_axis='time', y_axis='hz') 
    plt.colorbar()

In [3]:
session_info.show()

## 2.11.3. Gradient Boosting

### Contexto

El algoritmo Gradient Boosting es un método de ensemble que busca combinar varios clasificadores para construir un clasificador fuerte basado en boosting y aprendiendo en cada iteración.

**Algoritmo:**

1. **Inicialización**: 
    - Establecer un valor inicial para las predicciones. Para problemas de regresión, este podría ser simplemente el promedio de las salidas del conjunto de entrenamiento. Para problemas de clasificación, podría ser el logaritmo de la razón de probabilidades:

    $$ F_0(x) = \arg \min_{\rho} \sum_{i=1}^{N} L(y_i, \rho) $$
donde $ F_0 $ es la predicción inicial, $ N $ es el número total de muestras, $ L $ es la función de pérdida, $ y_i $ es el valor verdadero de la muestra $ i $, y $ \rho $ es el valor que minimiza la función de pérdida.

1. **Iterar**:
Desde $ m = 1 $ hasta $ M $, donde $ M $ es el número total de árboles (o modelos débiles) que deseamos entrenar:
    - **Calcular los residuos**. Para cada muestra $ i $ en el conjunto de entrenamiento, el residuo es la diferencia entre el valor verdadero y la predicción actual:
                                    $$ r_{im} = - \left[ \frac{\partial L(y_i, F(x_i))}{\partial F(x_i)} \right]_{F(x)=F_{m-1}(x)} $$

donde $ r_{im} $ es el residuo para la muestra $ i $ en la iteración $ m $.
- **Ajustar el modelo débil** $ h_m(x) $ a los residuos anteriores. Esto se hace comúnmente utilizando un árbol de decisión.
- **Encontrar el paso óptimo** $ \rho_m $ que minimiza la función de pérdida:
                                    $$ \rho_m = \arg \min_{\rho} \sum_{i=1}^{N} L \left( y_i, F_{m-1}(x_i) + \rho h_m(x_i) \right) $$
- **Actualizar el modelo**:
$$ F_m(x) = F_{m-1}(x) + \eta \cdot \rho_m h_m(x) $$

Aquí, $ \eta $ es el learning rate.

1. **Obtener el modelo final**:

    $$ F(x) = F_M(x) $$

    donde $ F(x) $ es la combinación final de todos los modelos débiles con sus respectivos pasos óptimos.

### Ejemplo

El conjunto de datos **Digits Audio**:

#### Manipulación de Datos y AED

In [3]:
file_path = '../03_Data/recordings/'
data = []

for i in os.listdir(file_path):
    try:
        x, sr = librosa.load(os.path.join(file_path, i), sr=None)
        data.append(x)
        print(f"Loaded: {i}")
    except Exception as e:
        print("")

Loaded: 0_george_0.wav
Loaded: 0_george_1.wav
Loaded: 0_george_10.wav
Loaded: 0_george_11.wav
Loaded: 0_george_12.wav
Loaded: 0_george_13.wav
Loaded: 0_george_14.wav
Loaded: 0_george_15.wav
Loaded: 0_george_16.wav
Loaded: 0_george_17.wav
Loaded: 0_george_18.wav
Loaded: 0_george_19.wav
Loaded: 0_george_2.wav
Loaded: 0_george_20.wav
Loaded: 0_george_21.wav
Loaded: 0_george_22.wav
Loaded: 0_george_23.wav
Loaded: 0_george_24.wav
Loaded: 0_george_25.wav
Loaded: 0_george_26.wav
Loaded: 0_george_27.wav
Loaded: 0_george_28.wav
Loaded: 0_george_29.wav
Loaded: 0_george_3.wav
Loaded: 0_george_30.wav
Loaded: 0_george_31.wav
Loaded: 0_george_32.wav
Loaded: 0_george_33.wav
Loaded: 0_george_34.wav
Loaded: 0_george_35.wav
Loaded: 0_george_36.wav
Loaded: 0_george_37.wav
Loaded: 0_george_38.wav
Loaded: 0_george_39.wav
Loaded: 0_george_4.wav
Loaded: 0_george_40.wav
Loaded: 0_george_41.wav
Loaded: 0_george_42.wav
Loaded: 0_george_43.wav
Loaded: 0_george_44.wav
Loaded: 0_george_45.wav
Loaded: 0_george_46.w

### Transformación

In [4]:
# Aplicación de Transformada de Fourier a frecuencias de sonido
data_tf=[]
for i in range(len(data)):
    data_tf.append(abs(librosa.stft(data[i]).mean(axis = 1).T))
data_tf= np.array(data_tf)

In [6]:
df = pd.DataFrame(data_tf)
df["target"] = [i[0] for i in os.listdir(file_path)[:-1]]

### EDA

In [8]:
# Visualización de frecuencias de sonido
wavPlayer("../03_Data/recordings/9_yweweler_7.wav")

None

In [9]:
data_tf.shape

(3000, 1025)

In [10]:
# Preparación de X y y
X = df[[x for x in df.columns if x != "target"]]
y = df["target"]

In [11]:
y.value_counts()

target
0    300
1    300
2    300
3    300
4    300
5    300
6    300
7    300
8    300
9    300
Name: count, dtype: int64

### Separación de sets

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y ,test_size=0.25)

## Modelado

### Gradient Boosting

#### Modelado

**Espacio Hiperparametral:**

1. `n_estimators`: Es el número de etapas de refuerzo que se ejecutarán. Es básicamente el número de árboles en el ensemble.
2. `max_depth`: Profundidad máxima de los estimadores individuales. Limita el número de nodos en el árbol.
3. `min_samples_split`: Es el número mínimo de muestras requeridas para dividir un nodo interno.
4. `min_samples_leaf`: El número mínimo de muestras requeridas para ser un nodo hoja.
5. `max_features`: Es el número de características a considerar cuando se busca la mejor división. `sqrt` y `log2` son opciones populares aparte de `auto`.

In [15]:
# Espacio hiperparametral
space = {
    'n_estimators': hp.choice('n_estimators', np.arange(50, 501, 50, dtype=int)),
    'max_depth': hp.choice('max_depth', np.arange(5, 31, 1, dtype=int)),
    'min_samples_split': hp.uniform('min_samples_split', 0.01, 1.0),
    'min_samples_leaf': hp.uniform('min_samples_leaf', 0.01, 0.5),
    'max_features': hp.choice('max_features', ['sqrt', 'log2']),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.2))
}

def objective(params):
    model = GradientBoostingClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)  # Usamos datos de validación en lugar de entrenamiento para la evaluación
    score = accuracy_score(y_test, preds)
    return {'loss': -score, 'status': STATUS_OK}

trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=10, trials=trials)
best_params = space_eval(space, best)

print("Mejores parámetros:", best_params)

100%|██████████| 10/10 [03:57<00:00, 23.71s/trial, best loss: -0.896]
Mejores parámetros: {'learning_rate': 0.07065961788783227, 'max_depth': 17, 'max_features': 'log2', 'min_samples_leaf': 0.06070060144982439, 'min_samples_split': 0.5451918460345738, 'n_estimators': 500}


In [16]:
gbm = GradientBoostingClassifier(**best_params)
gbm.fit(X_train, y_train)

GradientBoostingClassifier(learning_rate=0.07065961788783227, max_depth=17,
                           max_features='log2',
                           min_samples_leaf=0.06070060144982439,
                           min_samples_split=0.5451918460345738,
                           n_estimators=500)

In [18]:
get_cv_scores_report_classification(gbm,X_test,y_test,5)

Accuracy: 0.7941 (+/- 0.0711)


In [20]:
y_pred = gbm.predict(X_test)

# Crear matriz de confusión
cm = confusion_matrix(y_test, y_pred)

# Crear la visualización con Plotly
fig = ff.create_annotated_heatmap(z=cm, colorscale='Viridis')
fig.update_layout(title_text='Matriz de Confusión')
fig.show()

In [240]:
pd.to_pickle(gbm,'../02_Codes/01_Models/gbm.pkl')

In [241]:
gbm = pd.read_pickle('../02_Codes/01_Models/gbm.pkl')